In [13]:
import pandas as pd
import numpy as np
import re

# Load input dataset (raw user form only)
form_df = pd.read_csv('user profile form (Responses).csv')

# Extract English text from bilingual values (format: "English | عربي")
def extract_english(text):
    if pd.isna(text): return text
    return str(text).split('|')[0].strip()

# Map month names (English + Arabic) to numeric values
month_map = {
    'January': 1, 'يناير': 1,
    'February': 2, 'فبراير': 2,
    'March': 3, 'مارس': 3,
    'April': 4, 'أبريل': 4, 'ابريل': 4,
    'May': 5, 'مايو': 5,
    'June': 6, 'يونيو': 6,
    'July': 7, 'يوليو': 7,
    'August': 8, 'أغسطس': 8, 'اغسطس': 8,
    'September': 9, 'سبتمبر': 9,
    'October': 10, 'أكتوبر': 10, 'اكتوبر': 10,
    'November': 11, 'نوفمبر': 11,
    'December': 12, 'ديسمبر': 12
}

# Convert month text into numeric value (1-12)
def clean_month(text):
    if pd.isna(text): return np.nan
    text_str = str(text).strip()
    for k, v in month_map.items():
        if k.lower() in text_str.lower():
            return v
    return np.nan

# Convert Arabic digits -> English digits and extract numeric value
def clean_arabic_numerals(text):
    if pd.isna(text): return text
    text = str(text)
    arabic_to_english = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
    text = text.translate(arabic_to_english)
    text = re.sub(r'[^\d.-]', '', text)
    try:
        return float(text)
    except:
        return np.nan

# Normalize year values
def clean_year(year_val):
    val = clean_arabic_numerals(year_val)
    if pd.isna(val): return 2026
    if val >= 1445 and val <= 1450: return 2026
    if val < 2000: return 2026
    return int(val)

# Calculate expected cost based on KSA electricity tariffs
def estimate_cost(kwh):
    if pd.isna(kwh): return np.nan
    if kwh <= 6000:
        return kwh * 0.18
    else:
        return (6000 * 0.18) + ((kwh - 6000) * 0.30)

records = []
user_id_start = 300
household_id_start = 3000
bill_id_start = 20000

user_month_year_seen = set()

for idx, row in form_df.iterrows():
    u_id = user_id_start + idx
    h_id = household_id_start + idx * 10

    region = extract_english(row.iloc[1])
    housing = extract_english(row.iloc[2])
    residents = clean_arabic_numerals(row.iloc[3])
    cooking = extract_english(row.iloc[4])
    ac = extract_english(row.iloc[5])
    insulation = True if extract_english(row.iloc[6]) == 'Yes' else False

    for b in range(6):
        month_idx = 7 + b * 4
        year_idx = 8 + b * 4
        kwh_idx = 9 + b * 4
        cost_idx = 10 + b * 4

        m_raw = row.iloc[month_idx]
        y_raw = row.iloc[year_idx]
        kwh_raw = row.iloc[kwh_idx]
        cost_raw = row.iloc[cost_idx]

        m = clean_month(m_raw)
        if pd.isna(m): continue

        y = clean_year(y_raw)
        kwh = clean_arabic_numerals(kwh_raw)
        cost = clean_arabic_numerals(cost_raw)

        if pd.isna(kwh) or pd.isna(cost): continue

        expected_cost = estimate_cost(kwh)
        difference = abs(cost - expected_cost)

        if difference > (expected_cost * 0.25) and difference > 200:
            continue

        if y > 2026 or (y == 2026 and m > 4):
            continue

        dup_key = (u_id, y, m)
        if dup_key in user_month_year_seen:
            continue
        user_month_year_seen.add(dup_key)

        season = 'Winter' if m in [12, 1, 2] else 'Rest_of_Year'

        records.append({
            'user_id': u_id,
            'household_id': h_id,
            'bill_id': bill_id_start,
            'region': region,
            'Housing_type': housing,
            'number_of_residents': residents,
            'cooking_fuel': cooking,
            'ac_type': ac,
            'has_insulation': insulation,
            'month': int(m),
            'year': int(y),
            'season': season,
            'total_kwh': round(kwh, 2),
            'total_cost_sar': round(cost, 2)
        })
        bill_id_start += 1

# Convert records into final DataFrame and export directly
out_df = pd.DataFrame(records)
output_file = 'user_profile.csv'
out_df.to_csv(output_file, index=False)